# Major Project Extension: ML-Enhanced Trust Scoring
## Forward-Secure Lattice-Based IBE with Behavioral Anomaly Detection for IoT

**Base:** Forward-Secure Lattice-Based IBE with Trust-Based Verification (Minor Project)

**Extension:** Replace static +1/-1 trust scoring with ML-driven behavioral anomaly detection using a **6-model ensemble**: Isolation Forest, Random Forest, SVM, KNN, Decision Tree, and MLP Neural Network.

---

### What This Notebook Demonstrates

| Section | Description |
|---------|-------------|
| **1. Setup** | Load the base fs-IBE system and ML modules |
| **2. Attack Simulation** | Generate 5 types of synthetic IoT attack traffic |
| **3. ML Model Training** | Train 6-model ensemble engine |
| **4. Visual Analysis** | Confusion matrices (all models), feature importance, trust trajectories |
| **5. Live Demo** | Real-time ML trust scoring on simulated queries |
| **6. Comparison** | Static trust vs ML-enhanced trust (metrics table + charts) |

---
## 1. Setup & Imports

In [1]:
import sys, os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# Register lattice modules for import
import lattice_infrastructure
sys.modules["LatticeCrypto"] = lattice_infrastructure

import lattice_infrastructure as P1
from forward_security import UserOps
from Trust_Model import (TrustManager, DilithiumStub, Query, QueryValidator,
                         FeatureExtractor, match_query_to_data)
from ml_trust_engine import MLTrustEngine, FEATURE_NAMES, SUPERVISED_MODEL_NAMES
from attack_simulator import (generate_full_dataset, save_dataset, load_dataset,
                              FEATURE_NAMES as ATK_FEATURES)

# Visualization
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 120})

print("All modules loaded successfully.")
print(f"Lattice Infrastructure: n={P1.LatticeParams().n}, q={P1.LatticeParams().q}")

All modules loaded successfully.
Lattice Infrastructure: n=16, q=3329


---
## 2. Attack Traffic Simulation

We generate **synthetic IoT traffic** with 5 attack types to train our ML models:

| Attack Type | Description | Key Behavioral Signal |
|---|---|---|
| **Replay** | Reuses old valid queries | High epoch mismatch rate |
| **Flood** | Overwhelms with query volume | Very high frequency, tiny intervals |
| **Epoch Manipulation** | Requests wrong/expired epochs | Very high epoch mismatch |
| **Sybil** | Multiple fake identities | High frequency + sig failures |
| **Probing** | Slow methodical exploration | Unusual payload sizes |

In [2]:
# Generate deployment-shaped temporal windows with correlated features.
# The default mix is intentionally imbalanced: mostly normal traffic with
# rare attacks and ambiguous boundary cases.
dataset = generate_full_dataset(n_ambiguous=5000)

# Save and load as DataFrame
save_dataset(dataset, "training_data.csv")
X, y, attack_types = load_dataset("training_data.csv")

df = pd.read_csv("training_data.csv")
print(f"Total samples: {len(df)}")
print(f"Normal: {(df['label']==0).sum()}, Malicious: {(df['label']==1).sum()}")
print(f"\nAttack type distribution:")
print(df['attack_type'].value_counts().to_string())

  Saved 40000 samples to training_data.csv
Total samples: 40000
Normal: 32171, Malicious: 7829

Attack type distribution:
attack_type
normal                      29250
ambiguous                    4864
sybil                         981
epoch_manipulation            979
flood                         978
replay                        975
probing                       968
noisy_normal                  750
noisy_ambiguous               136
noisy_probing                  32
noisy_replay                   25
noisy_flood                    22
noisy_epoch_manipulation       21
noisy_sybil                    19


### 2.1 Dataset Distribution Visualization

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart - attack type distribution
type_counts = df['attack_type'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6', '#1abc9c']
axes[0].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 10})
axes[0].set_title('Traffic Distribution by Type', fontsize=13, fontweight='bold')

# Bar chart - feature means by label
normal_means = df[df['label']==0][FEATURE_NAMES].mean()
attack_means = df[df['label']==1][FEATURE_NAMES].mean()
x_pos = np.arange(len(FEATURE_NAMES))
w = 0.35
axes[1].barh(x_pos - w/2, normal_means, w, label='Normal', color='#2ecc71', alpha=0.85)
axes[1].barh(x_pos + w/2, attack_means, w, label='Attack', color='#e74c3c', alpha=0.85)
axes[1].set_yticks(x_pos)
axes[1].set_yticklabels([f.replace('_', ' ').title() for f in FEATURE_NAMES], fontsize=9)
axes[1].set_xlabel('Mean Value')
axes[1].set_title('Feature Means: Normal vs Attack', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('dataset_distribution.png', bbox_inches='tight')
plt.show()
print("Saved: dataset_distribution.png")

Saved: dataset_distribution.png


---
## 3. ML Trust Engine Training

### 6-Model Ensemble Architecture

```
Query Features (8 dims)
        |
        v
  [StandardScaler]
        |
   +----+----+----+----+----+----+
   |    |    |    |    |    |    |
   v    v    v    v    v    v    v
 [IF] [RF] [SVM] [KNN] [DT] [MLP]
   |    |    |    |    |    |    |
   v    v    v    v    v    v    v
 Anom  Prob  Prob  Prob Prob Prob
 Score
   |    |    |    |    |    |    |
   +----+----+----+----+----+----+
                  |
                  v
       Weighted Majority Voting
       (is_anomaly, confidence)
```

| Model | Type | Description |
|---|---|---|
| **Isolation Forest** | Unsupervised | Detects novel/unknown attacks without labels |
| **Random Forest** | Supervised | Ensemble tree classifier, high accuracy |
| **SVM** | Supervised | Kernel-based margin classifier |
| **KNN** | Supervised | Instance-based distance classifier |
| **Decision Tree** | Supervised | Interpretable tree-based classifier |
| **MLP** | Supervised | Neural network deep learning classifier |

In [4]:
# Train the ML Trust Engine (all 6 models)
engine = MLTrustEngine(n_estimators_iso=100, n_estimators_rf=100)
metrics = engine.train(X, y, verbose=True)
engine.save_models("models")


  ML Trust Engine — Training (6 Models)
  Training samples: 32000
  Test samples: 8000
  Features: 8

  [1/6] Training Isolation Forest (unsupervised)...
  Isolation Forest — Acc: 0.8486, Prec: 0.5754, Rec: 0.8653, F1: 0.6912

  [2/6] Training Random Forest...
  Random Forest — Acc: 0.9018, Prec: 0.8750, Rec: 0.5811, F1: 0.6984

  [3/6] Training SVM...
  SVM — Acc: 0.8980, Prec: 0.9204, Rec: 0.5243, F1: 0.6680

  [4/6] Training KNN...
  KNN — Acc: 0.8909, Prec: 0.7676, Rec: 0.6347, F1: 0.6949

  [5/6] Training Decision Tree...
  Decision Tree — Acc: 0.8902, Prec: 0.8475, Rec: 0.5358, F1: 0.6565

  [6/6] Training MLP (Neural Network)...
  MLP (Neural Network) — Acc: 0.8995, Prec: 0.8414, Rec: 0.5996, F1: 0.7002

  === Combined Ensemble Performance (6 Models) ===
  Accuracy:  0.8954
  Precision: 0.7544
  Recall:    0.6903
  F1 Score:  0.7209

  Confusion Matrix (Combined Ensemble):
    TN=6082  FP= 352
    FN= 485  TP=1081

  Feature Importance (Random Forest):
    epoch_mismatch_rate  

---
## 4. Visual Analysis

### 4.1 Confusion Matrix Heatmaps (All 6 Models + Combined)

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_test_scaled = engine.scaler.transform(X_test)

# Get predictions from all models
iso_pred = (engine.isolation_forest.predict(X_test_scaled) == -1).astype(int)
rf_pred = engine.random_forest.predict(X_test_scaled)
svm_pred = engine.svm.predict(X_test_scaled)
knn_pred = engine.knn.predict(X_test_scaled)
dt_pred = engine.decision_tree.predict(X_test_scaled)
mlp_pred = engine.mlp.predict(X_test_scaled)

# Get probabilities for combined prediction
iso_scores = -engine.isolation_forest.score_samples(X_test_scaled)
supervised_probas = {
    'random_forest': engine.random_forest.predict_proba(X_test_scaled)[:, 1],
    'svm': engine.svm.predict_proba(X_test_scaled)[:, 1],
    'knn': engine.knn.predict_proba(X_test_scaled)[:, 1],
    'decision_tree': engine.decision_tree.predict_proba(X_test_scaled)[:, 1],
    'mlp': engine.mlp.predict_proba(X_test_scaled)[:, 1],
}
supervised_preds = {
    'random_forest': rf_pred, 'svm': svm_pred, 'knn': knn_pred,
    'decision_tree': dt_pred, 'mlp': mlp_pred,
}
combined = engine._combine_predictions_multi(iso_pred, iso_scores, supervised_preds, supervised_probas)

# Plot confusion matrices: 2 rows x 4 columns
all_preds = [
    (iso_pred, 'Isolation Forest'), (rf_pred, 'Random Forest'),
    (svm_pred, 'SVM'), (knn_pred, 'KNN'),
    (dt_pred, 'Decision Tree'), (mlp_pred, 'MLP (Neural Net)'),
    (combined, 'Combined Ensemble'),
]
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes_flat = axes.flatten()

for idx, (pred, title) in enumerate(all_preds):
    ax = axes_flat[idx]
    cm_i = confusion_matrix(y_test, pred)
    im = ax.imshow(cm_i, cmap='Blues', interpolation='nearest')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Normal','Attack'])
    ax.set_yticklabels(['Normal','Attack'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm_i[i,j]), ha='center', va='center',
                    fontsize=16, fontweight='bold',
                    color='white' if cm_i[i,j] > cm_i.max()/2 else 'black')
    plt.colorbar(im, ax=ax, shrink=0.8)

# Hide the unused 8th subplot
axes_flat[7].set_visible(False)

plt.suptitle('Confusion Matrices — All 6 Models + Combined Ensemble', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()
print("Saved: confusion_matrices.png")

Saved: confusion_matrices.png


### 4.2 Feature Importance (Random Forest)

In [6]:
importances = engine.random_forest.feature_importances_
sorted_idx = np.argsort(importances)
labels = [FEATURE_NAMES[i].replace('_', ' ').title() for i in sorted_idx]

fig, ax = plt.subplots(figsize=(10, 6))
colors_grad = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(sorted_idx)))
bars = ax.barh(range(len(sorted_idx)), importances[sorted_idx], color=colors_grad)
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('Feature Importance for Attack Detection\n(Random Forest Classifier)',
             fontsize=14, fontweight='bold')

for i, (val, bar) in enumerate(zip(importances[sorted_idx], bars)):
    ax.text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=10, fontweight='bold')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()
print("Saved: feature_importance.png")

Saved: feature_importance.png


### 4.3 Trust Score Trajectory Over Time

This plot shows how trust scores evolve for different device behaviors:
- **Legitimate device**: Trust score climbs and stays high
- **Flood attacker**: Trust rapidly drops below zero (blocked)
- **Replay attacker**: Trust degrades as epoch mismatches accumulate

In [7]:
# Simulate trust score trajectories
np.random.seed(42)
n_steps = 50

def simulate_trajectory(engine, behavior='normal'):
    tm = TrustManager(ml_engine=engine)
    scores = [tm.get_score('dev')]
    for step in range(n_steps):
        if behavior == 'normal':
            features = {
                'query_frequency': np.random.uniform(2, 6),
                'epoch_mismatch_rate': np.random.uniform(0, 0.03),
                'sig_failure_rate': 0.0,
                'inter_query_interval': np.random.uniform(5, 20),
                'payload_size_deviation': np.random.uniform(0, 30),
                'time_of_day_sin': 0.5, 'time_of_day_cos': 0.866,
                'query_burst_count': np.random.choice([1, 2]),
            }
        elif behavior == 'flood':
            features = {
                'query_frequency': np.random.uniform(40, 90),
                'epoch_mismatch_rate': np.random.uniform(0, 0.15),
                'sig_failure_rate': np.random.uniform(0.1, 0.3),
                'inter_query_interval': np.random.uniform(0.01, 0.3),
                'payload_size_deviation': np.random.uniform(0, 20),
                'time_of_day_sin': 0.5, 'time_of_day_cos': 0.866,
                'query_burst_count': np.random.randint(15, 45),
            }
        elif behavior == 'replay':
            features = {
                'query_frequency': np.random.uniform(8, 18),
                'epoch_mismatch_rate': np.random.uniform(0.4, 0.7),
                'sig_failure_rate': np.random.uniform(0, 0.05),
                'inter_query_interval': np.random.uniform(1, 4),
                'payload_size_deviation': np.random.uniform(0, 15),
                'time_of_day_sin': 0.5, 'time_of_day_cos': 0.866,
                'query_burst_count': np.random.randint(3, 8),
            }
        tm.ml_score('dev', features)
        scores.append(tm.get_score('dev'))
    return scores

traj_normal = simulate_trajectory(engine, 'normal')
traj_flood = simulate_trajectory(engine, 'flood')
traj_replay = simulate_trajectory(engine, 'replay')

fig, ax = plt.subplots(figsize=(12, 6))
steps = range(n_steps + 1)
ax.plot(steps, traj_normal, 'g-o', markersize=4, label='Legitimate IoT Device', linewidth=2)
ax.plot(steps, traj_flood, 'r-s', markersize=4, label='Flood Attacker', linewidth=2)
ax.plot(steps, traj_replay, color='orange', marker='^', markersize=4, label='Replay Attacker', linewidth=2)
ax.axhline(y=0, color='black', linestyle='--', linewidth=1.5, alpha=0.7, label='Trust Threshold (score=0)')
ax.fill_between(steps, -15, 0, alpha=0.08, color='red')
ax.text(n_steps*0.8, -1.5, 'BLOCKED ZONE', fontsize=11, color='red', fontweight='bold', alpha=0.7)
ax.set_xlabel('Query Number', fontsize=12)
ax.set_ylabel('Trust Score', fontsize=12)
ax.set_title('Trust Score Evolution: ML-Enhanced Behavioral Detection',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='center left')
ax.set_xlim(0, n_steps)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('trust_trajectory.png', bbox_inches='tight')
plt.show()
print("Saved: trust_trajectory.png")

Saved: trust_trajectory.png


---
## 5. Static vs ML-Enhanced Trust: Full Comparison

### Running the comparison benchmark

In [8]:
from ml_simulation import run_ml_trust_comparison

results = run_ml_trust_comparison(n=64, num_queries=100, num_attacks=100)

# Print results table
s, m = results['static'], results['ml']
print("=" * 65)
print("  STATIC vs ML-ENHANCED TRUST SCORING")
print("=" * 65)
print(f"  {'Metric':<25} {'Static':>15} {'ML-Enhanced':>15}")
print("  " + "-" * 55)
for k in ['accuracy','precision','recall','f1','fpr']:
    print(f"  {k:<25} {s[k]:>15.4f} {m[k]:>15.4f}")
print("  " + "-" * 55)
print(f"  {'TP (attacks caught)':<25} {s['tp']:>15} {m['tp']:>15}")
print(f"  {'FP (false alarms)':<25} {s['fp']:>15} {m['fp']:>15}")
print(f"  {'TN (legit accepted)':<25} {s['tn']:>15} {m['tn']:>15}")
print(f"  {'FN (attacks missed)':<25} {s['fn']:>15} {m['fn']:>15}")
print("  " + "-" * 55)
avg_inference = results['ml_avg_inference_ms']
if avg_inference is None:
    print("  ML Avg Inference: not measured")
else:
    print(f"  ML Avg Inference: {avg_inference:.2f} ms")
print("=" * 65)

  STATIC vs ML-ENHANCED TRUST SCORING
  Metric                             Static     ML-Enhanced
  -------------------------------------------------------
  accuracy                           0.6000          0.9000
  precision                          1.0000          1.0000
  recall                             0.2000          0.8000
  f1                                 0.3333          0.8889
  fpr                                0.0000          0.0000
  -------------------------------------------------------
  TP (attacks caught)                    20              80
  FP (false alarms)                       0               0
  TN (legit accepted)                   100             100
  FN (attacks missed)                    80              20
  -------------------------------------------------------
  ML Avg Inference: not measured


### 5.1 Comparison Bar Chart

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metrics comparison
metrics_keys = ['accuracy', 'precision', 'recall', 'f1']
static_vals = [results['static'][k] for k in metrics_keys]
ml_vals = [results['ml'][k] for k in metrics_keys]

x = np.arange(len(metrics_keys))
w = 0.3
axes[0].bar(x - w/2, static_vals, w, label='Static Trust', color='#3498db', alpha=0.85)
axes[0].bar(x + w/2, ml_vals, w, label='ML-Enhanced', color='#2ecc71', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels([k.upper() for k in metrics_keys], fontsize=11)
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel('Score')
axes[0].set_title('Detection Performance Comparison', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
for i, (sv, mv) in enumerate(zip(static_vals, ml_vals)):
    axes[0].text(i - w/2, sv + 0.02, f'{sv:.2f}', ha='center', fontsize=9, fontweight='bold')
    axes[0].text(i + w/2, mv + 0.02, f'{mv:.2f}', ha='center', fontsize=9, fontweight='bold')

# Confusion matrix counts
labels = ['TP\n(caught)', 'TN\n(legit OK)', 'FP\n(false alarm)', 'FN\n(missed)']
static_cm = [s['tp'], s['tn'], s['fp'], s['fn']]
ml_cm = [m['tp'], m['tn'], m['fp'], m['fn']]
x2 = np.arange(4)
axes[1].bar(x2 - w/2, static_cm, w, label='Static', color='#3498db', alpha=0.85)
axes[1].bar(x2 + w/2, ml_cm, w, label='ML-Enhanced', color='#2ecc71', alpha=0.85)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(labels, fontsize=10)
axes[1].set_ylabel('Count')
axes[1].set_title('Confusion Matrix Comparison', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
for i, (sv, mv) in enumerate(zip(static_cm, ml_cm)):
    axes[1].text(i - w/2, sv + 0.3, str(sv), ha='center', fontsize=10, fontweight='bold')
    axes[1].text(i + w/2, mv + 0.3, str(mv), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('comparison_chart.png', bbox_inches='tight')
plt.show()
print("Saved: comparison_chart.png")

Saved: comparison_chart.png


### 5.2 ML Model Training Performance (All 6 Models)

In [10]:
tm = results['ml_train_metrics']

# All model keys and display names
model_keys = ['iso', 'random_forest', 'svm', 'knn', 'decision_tree', 'mlp', 'combined']
model_names = ['Isolation\nForest', 'Random\nForest', 'SVM', 'KNN', 'Decision\nTree', 'MLP', 'Combined\nEnsemble']
acc = [tm[f'{k}_accuracy'] for k in model_keys]
prec = [tm[f'{k}_precision'] for k in model_keys]
rec = [tm[f'{k}_recall'] for k in model_keys]
f1s = [tm[f'{k}_f1'] for k in model_keys]

fig, ax = plt.subplots(figsize=(16, 7))
x = np.arange(len(model_names))
w = 0.18
ax.bar(x - 1.5*w, acc, w, label='Accuracy', color='#2ecc71')
ax.bar(x - 0.5*w, prec, w, label='Precision', color='#3498db')
ax.bar(x + 0.5*w, rec, w, label='Recall', color='#f39c12')
ax.bar(x + 1.5*w, f1s, w, label='F1 Score', color='#e74c3c')
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('ML Model Performance Comparison — All 6 Models + Combined Ensemble',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add value labels on the F1 bars
for i, v in enumerate(f1s):
    ax.text(i + 1.5*w, v + 0.01, f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('model_performance.png', bbox_inches='tight')
plt.show()
print("Saved: model_performance.png")

# Print per-model metrics table
print("\n" + "=" * 70)
print(f"  {'Model':<22} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("  " + "-" * 65)
names_flat = ['Isolation Forest', 'Random Forest', 'SVM', 'KNN', 'Decision Tree', 'MLP', 'Combined Ensemble']
for i, k in enumerate(model_keys):
    print(f"  {names_flat[i]:<22} {acc[i]:>10.4f} {prec[i]:>10.4f} {rec[i]:>10.4f} {f1s[i]:>10.4f}")
print("=" * 70)

Saved: model_performance.png

  Model                    Accuracy  Precision     Recall         F1
  -----------------------------------------------------------------
  Isolation Forest           0.8489     0.5760     0.8640     0.6912
  Random Forest              0.9018     0.8707     0.5849     0.6998
  SVM                        0.8980     0.9204     0.5243     0.6680
  KNN                        0.8909     0.7676     0.6347     0.6949
  Decision Tree              0.8902     0.8475     0.5358     0.6565
  MLP                        0.8995     0.8414     0.5996     0.7002
  Combined Ensemble          0.8951     0.7533     0.6903     0.7204


---
## 6. Live Demo: Real-Time ML Trust Evaluation

Testing the ML engine on sample query feature vectors:

In [11]:
test_cases = [
    ("Normal IoT Sensor", {
        'query_frequency': 3, 'epoch_mismatch_rate': 0.0, 'sig_failure_rate': 0.0,
        'inter_query_interval': 12.0, 'payload_size_deviation': 5,
        'time_of_day_sin': 0.5, 'time_of_day_cos': 0.866, 'query_burst_count': 1}),
    ("Flood Attack", {
        'query_frequency': 75, 'epoch_mismatch_rate': 0.1, 'sig_failure_rate': 0.2,
        'inter_query_interval': 0.05, 'payload_size_deviation': 10,
        'time_of_day_sin': 0.5, 'time_of_day_cos': 0.866, 'query_burst_count': 35}),
    ("Replay Attack", {
        'query_frequency': 12, 'epoch_mismatch_rate': 0.6, 'sig_failure_rate': 0.0,
        'inter_query_interval': 2.0, 'payload_size_deviation': 5,
        'time_of_day_sin': 0.5, 'time_of_day_cos': 0.866, 'query_burst_count': 5}),
    ("Epoch Manipulation", {
        'query_frequency': 5, 'epoch_mismatch_rate': 0.85, 'sig_failure_rate': 0.05,
        'inter_query_interval': 5, 'payload_size_deviation': 40,
        'time_of_day_sin': 0.5, 'time_of_day_cos': 0.866, 'query_burst_count': 2}),
    ("Sybil Attack", {
        'query_frequency': 25, 'epoch_mismatch_rate': 0.25, 'sig_failure_rate': 0.35,
        'inter_query_interval': 0.5, 'payload_size_deviation': 80,
        'time_of_day_sin': 0.5, 'time_of_day_cos': 0.866, 'query_burst_count': 10}),
]

print("=" * 70)
print(f"  {'Scenario':<22} {'Anomaly?':<10} {'Confidence':<12} {'Score':<10} {'Verdict'}")
print("=" * 70)
for name, features in test_cases:
    is_anom, conf, score = engine.evaluate(features)
    verdict = "BLOCK" if is_anom else "ALLOW"
    symbol = "X" if is_anom else "OK"
    print(f"  {name:<22} {str(is_anom):<10} {conf:<12.4f} {score:<10.4f} [{symbol}] {verdict}")
print("=" * 70)

  Scenario               Anomaly?   Confidence   Score      Verdict
  Normal IoT Sensor      False      0.9669       0.1473     [OK] ALLOW
  Flood Attack           True       0.7576       0.7511     [X] BLOCK
  Replay Attack          True       0.9809       0.8729     [X] BLOCK
  Epoch Manipulation     True       0.9714       0.8701     [X] BLOCK
  Sybil Attack           True       0.9668       0.9046     [X] BLOCK


---
## 7. Summary & Key Findings

### What We Built
| Component | Description |
|---|---|
| **Attack Simulator** | 5 attack types, 40,000 temporally correlated synthetic windows |
| **ML Trust Engine** | 6-model ensemble: Isolation Forest + Random Forest + SVM + KNN + Decision Tree + MLP |
| **Feature Extraction** | 8 behavioral features per query |
| **Trust Integration** | ML scores drive dynamic trust adjustments via weighted majority voting |

### Models Used
| Model | Type | Why Selected |
|---|---|---|
| **Isolation Forest** | Unsupervised | Catches novel/unknown attack patterns without labels |
| **Random Forest** | Supervised | High accuracy ensemble tree classifier |
| **SVM** | Supervised | Kernel-based margin classifier, widely used in IDS research |
| **KNN** | Supervised | Instance-based distance classifier, low complexity for IoT |
| **Decision Tree** | Supervised | Interpretable tree-based classifier |
| **MLP** | Supervised | Neural network deep learning representative |

### Key Results
The end-to-end comparison uses matched independent episodes. Each query is
processed through the normal validator path, and the attack histories are
created by actual signed or invalid queries rather than direct feature injection.

| Metric | Static | ML-enhanced (6-model) |
|---|---:|---:|
| Accuracy | 60% | 90% |
| Precision | 100% | 100% |
| Recall | 20% | 80% |
| F1 score | 0.333 | 0.889 |
| False-positive rate | 0% | 0% |

The 6-model ensemble detects the valid-signature flood, wrong-epoch, and replay
behaviors, while static trust detects only invalid signatures. The weighted
majority voting across all supervised models provides more robust decisions
than a dual-model approach.

### Limitations and Next Steps
- Add explicit Sybil and stealth behavior tracking to the query validator.
- Evaluate with multiple random seeds and an independent held-out traffic trace.
- Report confidence intervals and per-attack precision/recall.
- Explore model-specific weighting based on per-attack-type performance.

### Novelty
> No existing paper combines **lattice-based fs-IBE** with **ML-driven trust scoring**.
> This extension directly fulfills the Future Work section of our minor project report:
> *"We aim to improve the trust model using lightweight machine learning for smarter anomaly detection."*

### Files Structure
```
Project/
|-- lattice_infrastructure.py   # P1: Lattice primitives
|-- forward_security.py         # P2: fs-IBE Encrypt/Decrypt
|-- Trust_Model.py              # P3: Enhanced trust + feature extraction
|-- simulation.py               # P4: Full pipeline benchmarking
|-- attack_simulator.py         # NEW: Synthetic attack generation
|-- ml_trust_engine.py          # NEW: 6-model ML engine
|-- ml_simulation.py            # NEW: ML comparison benchmarks
|-- models/                     # Saved ML models (joblib)
|-- training_data.csv           # Generated training dataset
|-- Forward_Secure_Lattice_IBE.ipynb     # Minor project notebook
|-- Major_Project_ML_Trust_Scoring.ipynb # THIS notebook (Major project)
```